In [1]:
import findspark
import os
#findspark.init() 
SPARK_HOME='/opt/cloudera/parcels/CDH/lib/spark'
# SPARK_HOME='/home/qiany/.conda/envs/py37'
# os.environ['SPARK_HOME'] = '/home/qiany/.conda/envs/py37'
findspark.init(SPARK_HOME)

In [2]:
import time
import math
import copy
import csv
import json
import os
import codecs
import subprocess
#from hdfs import InsecureClient
import numpy as np
#from pyspark import SparkContext
from pyspark import SQLContext
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.functions import create_map
from pyspark.sql.functions import array_union,flatten,array_sort,coalesce,broadcast,collect_list, collect_set, udf, array_remove, log, lit, first, col, array, sort_array,split, explode, desc, asc, row_number,isnan, when, count
from pyspark.sql.types import *
import rtree
from pyspark.sql import Window
#import igraph
#from igraph import Graph
import geofeather
from pyspark.storagelevel import StorageLevel

In [3]:
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410_final/ALS_L1B_20190410T174554_181213_small_crop_as_100000.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410_final/ALS_L1B_20190410T174554_181213_3_crop_as_100000.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410_final/OR22_01per_as_10000.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T183726_185444_1/ALS_L1B_20190410T183726_185444_1.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T153919_155000_1/ALS_L1B_20190410T153919_155000_1.off
# /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T174554_181213/ALS_L1B_20190410T174554_181213.off
tin_file = input("Here is a programe to compute the Forman gradient, please input the absolute or relative path in local disk to your TIN file:")
# tin_file = '/local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410_final/ALS_L1B_20190410T174554_181213_3_crop_as_100000.off'

print("\n********************")

# get the directory to the TIN file
tin_directory = os.path.dirname(tin_file)
print("tin_directory: ", tin_directory)

# ALS_L1B_20190410T153919_155000_1_pers_025
# ALS_L1B_20190410T174554_181213_pers_025
directory = input("What is the directory (e.g., hdfs_data, ALS_L1B_20190410T183726_185444_1_pers_025):") or "2"

# directory_type = input("Is the data stored in hdfs(0) or SigSpatial(1) or seaice(2) or seaice_11122025(3):") or "2"
# if directory_type == '0':
#     directory = 'hdfs_data'
# elif directory_type == '1':
#     directory = 'SigSpatial_data'
# elif directory_type == '2':
#     directory = 'seaice_data'
# else:
#     directory = 'seaice_11122025'
    
# get the basename to the TIN file
tin_basename = os.path.basename(tin_file) # input_vertices_2.off
print("tin_basename: ", tin_basename)

# get the filename of the TIN file
tin_filename = os.path.splitext(tin_basename)[0] # input_vertices_2
print("tin_filename: ", tin_filename)

# get the type of TIN file: off, tri, etc
tin_extension = os.path.splitext(tin_basename)[1] # .off
print("tin_extension: ", tin_extension)
print("\n********************")

Here is a programe to compute the Forman gradient, please input the absolute or relative path in local disk to your TIN file: /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T174554_181213/ALS_L1B_20190410T174554_181213.off



********************
tin_directory:  /local/data/yuehui/pyspark/data/Part3_TopoSim/ALS_L1B_20190410T174554_181213


What is the directory (e.g., hdfs_data, ALS_L1B_20190410T183726_185444_1_pers_025): ALS_L1B_20190410T174554_181213_pers_025


tin_basename:  ALS_L1B_20190410T174554_181213.off
tin_filename:  ALS_L1B_20190410T174554_181213
tin_extension:  .off

********************


In [4]:
# filtra is the order of each vertex, the order is obtained by ranking elevation values of vertices
filtra = input("Do you have filtration data?") or "yes"
# filtra = 'yes'

if filtra.lower() == 'no':    
    Basic_Data = input("Do you have basic pts and tri data?")
    
# graph = input("Is this graph VE (0) or ET (1)?") or "1"
Num_executor = input("spark.executor.instances:") or "16"
Num_core_per_executor = input("spark.executor.cores:") or "5"
Memory_executor = input("spark.executor.memory? Please end with 'g':") or "56g"
MemoryOverhead_executor = input("spark.executor.memoryOverhead? Please end with 'g':") or "24g"

# Num_core_per_driver = Num_core_per_executor
# Memory_driver = Memory_executor
# MemoryOverhead_driver = MemoryOverhead_executor

Num_core_per_driver = '5'
Memory_driver = '56g'
MemoryOverhead_driver = '24g'

Num_shuffle_partitions = input("spark.sql.shuffle.partitions:") or "160"

Do you have filtration data? yes
spark.executor.instances: 16
spark.executor.cores: 5
spark.executor.memory? Please end with 'g': 
spark.executor.memoryOverhead? Please end with 'g': 
spark.sql.shuffle.partitions: 160


In [5]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel
import geopandas as gpd
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, FloatType, ArrayType, MapType
from shapely.geometry import Point
from shapely.geometry import Polygon

from sedona.register import SedonaRegistrator
from sedona.core.SpatialRDD import SpatialRDD, PointRDD, CircleRDD, PolygonRDD, LineStringRDD
from sedona.core.enums import FileDataSplitter
from sedona.utils.adapter import Adapter
from sedona.core.spatialOperator import KNNQuery
from sedona.core.spatialOperator import JoinQuery
from sedona.core.spatialOperator import JoinQueryRaw
from sedona.core.spatialOperator import RangeQuery
from sedona.core.spatialOperator import RangeQueryRaw
from sedona.core.formatMapper.shapefileParser import ShapefileReader
from sedona.core.formatMapper import WkbReader
from sedona.core.formatMapper import WktReader
from sedona.core.formatMapper import GeoJsonReader
from sedona.sql.types import GeometryType
from sedona.core.enums import GridType
from sedona.core.SpatialRDD import RectangleRDD
from sedona.core.enums import IndexType
from sedona.core.geom.envelope import Envelope
from sedona.utils import SedonaKryoRegistrator, KryoSerializer

In [6]:
os.environ['PYSPARK_PYTHON'] = "./environment/bin/python"
#os.environ['PYSPARK_PYTHON'] = "/home/qiany/.conda/envs/py37/bin/python"
os.environ['YARN_CONF_DIR'] = "/opt/cloudera/parcels/CDH/lib/spark/conf/yarn-conf"

In [7]:
'''
spark.executor.cores: # Number of concurrent tasks an executor can run, euqals to the number of cores to use on each executor
spark.executor.instances: # Number of executors for the spark application
spark.executor.memory: # Amount of memory to use for each executor that runs the task
spark.executor.memoryOverhead:
spark.driver.cores: # Number of cores to use for the driver process; the default number is 1
spark.driver.memory: # Amount of memory to use for the driver
spark.driver.maxResultSize: to define the maximum limit of the total size of the serialized result that a driver can store for each Spark collect action
spark.default.parallelism: # Default number of partitions in RDDs returned by transformations like join, reduceByKey, and parallelize when not set by user. It can be set as spark.executor.instances * spark.executor.cores * 2
spark.sql.shuffle.partitions: determine how many partitions are used when data is shuffled between nodes, e.g., joins or aggregations. usually 1~5 times of executor.instances * executor.cores
spark.memory.storageFraction: determines the fraction of the heap space that is allocated to caching RDDs and DataFrames in memory.
spark.kryoserializer.buffer.max: determine the maximum of data that can be serialized at once; this must be larger than any object we attempt to serialize
spark.rpc.message.maxSize: # Maximum message size (in MiB) to allow in "control plane" communication; generally only applies to map output size information sent between executors and the driver. To communicate between the nodes, Spark uses a protocol called RPC (Remote Procedure Call), which sends messages back and forth. The spark.rpc.message.maxSize parameter limits how big these messages can be. 
spark.sql.broadcastTimeout: Spark will wait for this amount of time before giving up on broadcasting a table. Broadcasting can take a long time if the table is large or if there is a shuffle operation before it.
spark.sql.autoBroadcastJoinThreshold: Spark will broadcast a table to all worker nodes when performing a join if its size is less than this value; -1 means disabling broadcasting
'''

date = time.strftime("%m,%d,%Y")
date_name = date.split(',')[0] + date.split(',')[1] + date.split(',')[2]

hour = time.strftime("%H,%M")
hour_name = hour.split(',')[0] + hour.split(',')[1]

# set the Spark app name
spark_app_name = "Seaice_step1_save_Forman_" + tin_filename + '_' + date_name + '_' + hour_name
print("spark_app_name:", spark_app_name)

spark = SparkSession \
.builder \
.appName(spark_app_name) \
.master('yarn') \
.config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
.config("spark.kryo.registrator", SedonaKryoRegistrator.getName) \
.config('spark.jars','sedona-core-2.4_2.11-1.0.0-incubating.jar,sedona-sql-2.4_2.11-1.0.0-incubating.jar,sedona-python-adapter-2.4_2.11-1.0.0-incubating.jar,sedona-viz-2.4_2.11-1.0.0-incubating.jar,geotools-wrapper-geotools-24.0.jar,graphframes-0.8.0-spark2.4-s_2.11.jar') \
.config('spark.executor.cores', Num_core_per_executor) \
.config('spark.executor.instances', Num_executor) \
.config('spark.executor.memory', Memory_executor) \
.config('spark.executor.memoryOverhead', MemoryOverhead_executor) \
.config('spark.driver.cores', Num_core_per_driver) \
.config('spark.driver.memory', Memory_driver) \
.config('spark.driver.memoryOverhead', MemoryOverhead_driver) \
.config('spark.driver.maxResultSize', '0') \
.config('spark.dynamicAllocation.enabled', 'false') \
.config('spark.network.timeout', '10000001s') \
.config('spark.executor.heartbeatInterval', '10000000s') \
.config('spark.sql.shuffle.partitions', Num_shuffle_partitions) \
.config("spark.default.parallelism", '20') \
.config("spark.kryoserializer.buffer.max", "1024mb") \
.config('spark.rpc.message.maxSize', '256') \
.config("spark.sql.broadcastTimeout", "36000") \
.config("spark.sql.autoBroadcastJoinThreshold", "-1") \
.config("spark.python.profile", "true") \
.config("spark.eventLog.enabled", "true") \
.config('spark.yarn.dist.archives', '/local/data/yuehui/py37.tar.gz#environment') \
.getOrCreate()

spark_app_name: Seaice_step1_save_Forman_ALS_L1B_20190410T174554_181213_03162026_1630


In [8]:
import sys
sys.path.append("/local/data/yuehui/pyspark/Topo_Simplification/code")

In [9]:
import graphframes
from graphframes import GraphFrame
from graphframes import *
from graphframes.lib import Pregel

In [10]:
file_df_ver_order = directory + '/' + tin_filename + '_filtra_pts' + '.parquet'
df_ver_order = spark.read.parquet(file_df_ver_order)
df_ver_order.printSchema()

root
 |-- x: float (nullable = true)
 |-- y: float (nullable = true)
 |-- ele: float (nullable = true)
 |-- self_index: integer (nullable = true)
 |-- self_order: integer (nullable = true)



In [11]:
file_df_tri_order = directory + '/' + tin_filename + '_filtra_tri' + '.parquet'
df_tri_order = spark.read.parquet(file_df_tri_order)
df_tri_order.printSchema()

root
 |-- tri_order: integer (nullable = true)
 |-- r1: integer (nullable = true)
 |-- r2: integer (nullable = true)
 |-- r3: integer (nullable = true)
 |-- r1_ele: float (nullable = true)
 |-- r2_ele: float (nullable = true)
 |-- r3_ele: float (nullable = true)



In [12]:
# sort the extreme vertices of a triangle in ascending order, e.g., (2,5,3) -> (2,3,5)

'''
def get_tri_array(r1, r2, r3):
# get_multi_pt_index is used to obtain the adjacent vertexes index, including the vertex itself
# pt_list1: partial adjacent vertex indexes of join result 1
# pt_list2: partial adjacent vertex indexes of join result 2
    tri = [r1, r2, r3]    
    tri.sort(reverse=True)
    
    return tri

# convert a function to an udf and determine the return type
# https://spark.apache.org/docs/3.1.3/api/python/reference/api/pyspark.sql.functions.udf.html
get_tri_array_udf = udf(get_tri_array, ArrayType(IntegerType()))
df_tri_order = df_tri_order.withColumn("tri", get_tri_array_udf(df_tri_order.r1, df_tri_order.r2, df_tri_order.r3))
'''

df_tri_order = df_tri_order.withColumn("tri_origin", F.array("r1", "r2", "r3")).withColumn("tri_ele_origin", F.array("r1_ele", "r2_ele", "r3_ele"))
df_tri_order = df_tri_order.withColumn("tri", sort_array("tri_origin", False)).drop("tri_origin") # sort_array("tri", False), False means descending order
df_tri_order = df_tri_order.withColumn("tri_ele", sort_array("tri_ele_origin", False)).drop("tri_ele_origin") # sort_array("tri", False), False means descending order

df_tri_order.printSchema()
df_tri_order.rdd.getNumPartitions()
#df_tri_order.show()

root
 |-- tri_order: integer (nullable = true)
 |-- r1: integer (nullable = true)
 |-- r2: integer (nullable = true)
 |-- r3: integer (nullable = true)
 |-- r1_ele: float (nullable = true)
 |-- r2_ele: float (nullable = true)
 |-- r3_ele: float (nullable = true)
 |-- tri: array (nullable = false)
 |    |-- element: integer (containsNull = true)
 |-- tri_ele: array (nullable = false)
 |    |-- element: float (containsNull = true)



54

### get VT relation

In [13]:
# union and group vertices and get the preliminary VT relation
def grp_union(df_tri_order):
    '''
    df_tri_order: a DataFrame storing sorted extreme vertices of each triangle
    '''
    # groupby and collect_set
    df_tri_group1 = df_tri_order.groupBy('r1','r1_ele').agg(collect_list('tri').alias('multi_tri'), collect_list('tri_ele').alias('multi_tri_ele'))
    df_tri_group2 = df_tri_order.groupBy('r2','r2_ele').agg(collect_list('tri').alias('multi_tri'), collect_list('tri_ele').alias('multi_tri_ele'))
    df_tri_group3 = df_tri_order.groupBy('r3', 'r3_ele').agg(collect_list('tri').alias('multi_tri'), collect_list('tri_ele').alias('multi_tri_ele'))
    
    union12 = df_tri_group1.union(df_tri_group2) # the title of union12 will be the same as df_sj_group1
    union123 = union12.union(df_tri_group3) # the title of union123 will be the same as union12, so as df_sj_group1

    union123_group = union123.groupBy('r1','r1_ele').agg(collect_list('multi_tri').alias('multi_tri_list'), collect_list('multi_tri_ele').alias('multi_tri_ele_list'))
    
    # print("the schema of union123_group:", union123_group.printSchema())
    return union123_group

In [14]:
union123_group = grp_union(df_tri_order)
union123_group.printSchema()

root
 |-- r1: integer (nullable = true)
 |-- r1_ele: float (nullable = true)
 |-- multi_tri_list: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: array (containsNull = true)
 |    |    |    |-- element: integer (containsNull = true)
 |-- multi_tri_ele_list: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: array (containsNull = true)
 |    |    |    |-- element: float (containsNull = true)



In [15]:
# sort the triangles in the preliminary VT relation
def get_multi_tri_order(tri_list, tri_ele_list):
    '''
    get_multi_tri_order is used to obtain the adjacent triangles, the results are in ascending order, but the triangle itself is in descending
    e.g., [[3, 2, 1], [5, 3, 2], [5, 3, 1]]
    tri_list: an array of array of array of integer
    tri_ele_list: an array of array of array of float
    '''

    tri = []
    tri_ele = []
    # pt_list.append(pt_self) # if we don't calculate roughness, we don't need self vertex
    for i in range(len(tri_list)):
        for j in range(len(tri_list[i])):
            # if tri_list[i][j] not in tri: # we do not need to check if it exists in tri_list, because we can prove that each one is unique
            tri.append(tri_list[i][j]) # update() will add multiple elements to a set, while add() only add one element to a set   
            tri_ele.append(tri_ele_list[i][j])
    
    tri_copy = copy.deepcopy(tri) # deep copy tri
    tri.sort() # sort the list of array, e.g., tri=[[3, 2, 1], [5, 2, 1], [5, 3, 1]], after sorting, [[3, 2, 1], [5, 2, 1], [5, 3, 1]]
    
    tri_ele_sort = []
    # sort tri_ele according to the same rule as tri
    for i in range(len(tri)):
        index_in_tri_origin = tri_copy.index(tri[i])
        tri_ele_sort.append(tri_ele[index_in_tri_origin])
    
    return tri, tri_ele_sort

# convert a function to an udf and determine the return type
# https://spark.apache.org/docs/3.1.3/api/python/reference/api/pyspark.sql.functions.udf.html

# StructType for get_multi_tri_order        
get_multi_tri_order_schema = StructType([
    StructField("multi_tri_sort", ArrayType(ArrayType(IntegerType())),True), 
    StructField('multi_tri_ele_sort',ArrayType(ArrayType(FloatType())),True)
])

get_multi_tri_order_udf = udf(get_multi_tri_order, get_multi_tri_order_schema)

df_VT = union123_group.withColumn("multi_tri_order", get_multi_tri_order_udf(union123_group.multi_tri_list, union123_group.multi_tri_ele_list))
df_VT = df_VT.select(col("r1").alias("Ver"), col("multi_tri_order.multi_tri_sort").alias("VT_filtra"), col("multi_tri_order.multi_tri_ele_sort").alias("VT_ele"))
df_VT.printSchema()
print("number of partitions for df_VT:", df_VT.rdd.getNumPartitions())

root
 |-- Ver: integer (nullable = true)
 |-- VT_filtra: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- VT_ele: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: float (containsNull = true)

number of partitions for df_VT: 160


### obtain the lower star of each vertex

In [16]:
# obtain Lower Star
def get_LS(Ver, VT_filtra, VT_ele):
# get_LS is used to obtain the edges and triangles of LS in ascending order
# VV: an array of VV relation in ascending order
    VV = set(num for sublist in VT_filtra for num in sublist if num != Ver)
    VV = sorted(VV)
        
    LS_edge = []
    for i in range(len(VV)):
        if VV[i] < int(Ver):
            LS_edge.append([int(Ver),VV[i]])
        
    LS_tri = []
    LS_tri_ele = []
    
    for i in range(len(VT_filtra)):
        if Ver >= max(VT_filtra[i]): # Ver_ele is not the maximum vertex
            LS_tri.append(VT_filtra[i])
            LS_tri_ele.append(VT_ele[i])
                
    return LS_edge, LS_tri, LS_tri_ele

# StructType for get_multi_tri_order        
get_LS_schema = StructType([
    StructField("LS_edge", ArrayType(ArrayType(IntegerType())),True), 
    StructField("LS_tri_filtra", ArrayType(ArrayType(IntegerType())),True),
    StructField('LS_tri_ele',ArrayType(ArrayType(FloatType())),True)
])

get_LS_udf = udf(get_LS, get_LS_schema)

df_LS = df_VT.withColumn("LS", get_LS_udf(df_VT.Ver, df_VT.VT_filtra, df_VT.VT_ele))
df_LS = df_LS.select("Ver", "VT_filtra", col("LS.LS_edge").alias("LS_edge"), col("LS.LS_tri_filtra").alias("LS_tri_filtra"), col("LS.LS_tri_ele").alias("LS_tri_ele"))
df_LS.printSchema()
print("number of partitions for df_LS:", df_LS.rdd.getNumPartitions())
# df_LS_edge.show()

root
 |-- Ver: integer (nullable = true)
 |-- VT_filtra: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- LS_edge: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- LS_tri_filtra: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- LS_tri_ele: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: float (containsNull = true)

number of partitions for df_LS: 160


### compute Forman Gradient

In [17]:
from collections import deque

In [18]:
# get the Forman gradient from lower stars
def get_Forman(Ver, LS_edge, LS_filtra, LS_ele):
    '''
    get_Forman is used to compute Forman gradient
    LS_edge is triangles of lower star in ascending order
    LS_tri is triangles of lower star in ascending order
    '''
    # inner udf to get the number of unpaired edges for a triangle
    def num_unpaired_edges(tri_inner, crit_cell_inner, Forman_vec_pair_VE_inner, Forman_vec_pair_ET_inner):
        # compute the number of unpaired edges
        edge0 = [tri_inner[0], tri_inner[1]]
        edge1 = [tri_inner[0], tri_inner[2]]
        
        num_unpaired_edge = 2
        paired_edge = []
        for crit_inner_temp in crit_cell_inner:
            if len(crit_inner_temp) == 2: # crit_inner_temp is a critical edge
                paired_edge.append(crit_inner_temp)
                
        for pair_vec in Forman_vec_pair_VE_inner: # pair_vec is a vector, it can be (ver, edge) or (edge, tri)
            paired_edge.append(pair_vec[1])
            # if type(pair_vec[0]) != int and len(pair_vec[0]) == 2: # pair_vec[0] is a paired edge, like in (edge, tri)
                # paired_edge.append(pair_vec[0])
            # if len(pair_vec[1]) == 2: # pair_vec[1] is a paired edge, like in (ver, edge)
                # paired_edge.append(pair_vec[1])
        for pair_vec in Forman_vec_pair_ET_inner: # pair_vec is a vector, it can be (ver, edge) or (edge, tri)
            paired_edge.append(pair_vec[0])
                
        if edge0 in paired_edge:
            num_unpaired_edge = num_unpaired_edge - 1
        if edge1 in paired_edge:
            num_unpaired_edge = num_unpaired_edge - 1
            
        return num_unpaired_edge
    
    # inner udf to get the edge which is paired with a triangles
    def pair_an_edge_from_PQzero(alpha_inner, PQzero_inner):
    # alpha_inner is a triangle in [r0, r1, r2] format, and the vertext's elevation is in descending order
    # PQzero_inner is a queue storing all other edges
        alpha_e0 = [alpha_inner[0], alpha_inner[1]]
        alpha_e1 = [alpha_inner[0], alpha_inner[2]]
        for edge_inner in PQzero_inner:
            if edge_inner == alpha_e0:
                return edge_inner
            if edge_inner == alpha_e1:
                return edge_inner
            
    crit_cell = [] # crit_cell will store critical simplices
    # Forman_vec_pair = [] # Forman_vec_pair will store all Forman gradient pairs as a vector
    Forman_vec_pair_VE = []
    Forman_vec_pair_ET = []
    if (len(LS_edge)+len(LS_filtra)) == 0: # ver is a local minimum
        crit_cell.append([Ver]) # store minimum as an array [Ver] though it has only one element
    elif len(LS_edge) > 0:
        Forman_vec_pair_VE.append([Ver, LS_edge[0]]) # LS_edge[0] is the minimum edge since it is in ascending order
        
        # define two queues, PQzero storing all other edges, PQone storing all triangles which have only one unpaired edge
        PQzero = deque()
        PQone = deque()
        PQone_ele = deque()
        for i in range(1, len(LS_edge)): # add the other edges to PQzero
            PQzero.append(LS_edge[i])
            
        cell_temp_index_of_remove = []
        index_temp = 0
        for cell_temp in LS_filtra:
            if num_unpaired_edges(cell_temp, crit_cell, Forman_vec_pair_VE, Forman_vec_pair_ET) == 1:
                PQone.append(cell_temp) # storing cells with one unpaired edges, which are triangles in TIN
                PQone_ele.append(LS_ele[index_temp])
                # remove cell_temp from LS_filtra
                # LS_filtra.remove(cell_temp)
                cell_temp_index_of_remove.append(index_temp)
            index_temp += 1
            
        # remove cell_temp from LS_filtra and LS_ele
        for i in reversed(cell_temp_index_of_remove):
            del LS_filtra[i]
            del LS_ele[i]
                
        # sort PQone in ascending order, the initial queue is already sorted when creating
        # PQone = sorted(PQone) # sort() is not supported but we can use sorted()
        # PQone = deque(PQone) # after sorting, PQone one will be a list instead of a queue, we need to reconstruct this queue
        while len(PQone) > 0 or len(PQzero) > 0:
            while len(PQone) > 0:
                # alpha = PQone.popleft() # alpha is a triangle in the PQone
                # alpha should be a triangle with lower elevation when PQone have more than one triangle
                if len(PQone) > 1 and abs(PQone_ele[0][1]-PQone_ele[1][1]) < 0.000001 and PQone_ele[0][2] > PQone_ele[1][2]:
                    alpha = PQone[1] # alpha is the second triangle
                    del PQone[1]
                    del PQone_ele[1]
                else:
                    alpha = PQone.popleft() # alpha is the first triangle, which is the same as alpha=PQone[0], then del PQone[0]
                    del PQone_ele[0]
                
                if num_unpaired_edges(alpha, crit_cell, Forman_vec_pair_VE, Forman_vec_pair_ET) == 0:
                    PQzero.append(alpha)
                else:
                    pair_alpha = pair_an_edge_from_PQzero(alpha, PQzero) # pair_alpha is an edge, [pair_alpha, alpha] will be a new paired vector
                    # if pair_alpha in PQzero:
                    PQzero.remove(pair_alpha) # remove pair_alpha from PQzero
                    # Forman_vec_pair.append([pair_alpha, alpha]) # add the [pair_alpha, alpha], which is [edge, tri]
                    Forman_vec_pair_ET.append([pair_alpha, alpha]) # add the [pair_alpha, alpha], which is [edge, tri]
                    
                    beta_cell_temp_index_of_remove = []
                    beta_index_temp = 0
                    for cell_beta in LS_filtra: # this can be optimized
                        if cell_beta not in PQone and num_unpaired_edges(cell_beta, crit_cell, Forman_vec_pair_VE, Forman_vec_pair_ET) == 1:
                            PQone.append(cell_beta)
                            PQone_ele.append(LS_ele[beta_index_temp])
                            # remove cell_beta from LS_tri
                            # LS_tri.remove(cell_beta)
                            beta_cell_temp_index_of_remove.append(beta_index_temp)
                        beta_index_temp += 1
                    
                    # remove cell_beta from LS_filtra and LS_ele
                    for i in reversed(beta_cell_temp_index_of_remove):
                        del LS_filtra[i]
                        del LS_ele[i]
                    
                    PQone = sorted(PQone)
                    PQone = deque(PQone) # after sorting, PQone one will be a list instead of a queue, we need to reconstruct this queue
            if len(PQzero) > 0:
                gama = PQzero.popleft() # gama is a critical simplex
                crit_cell.append(gama)
                
                gama_cell_temp_index_of_remove = []
                gama_index_temp = 0
                for cell_gama in LS_filtra: # this can be optimized
                    if cell_gama not in PQone and num_unpaired_edges(cell_gama, crit_cell, Forman_vec_pair_VE, Forman_vec_pair_ET) == 1:
                        PQone.append(cell_gama)
                        PQone_ele.append(LS_ele[gama_index_temp])
                        # remove cell_gama from LS_tri
                        # LS_tri.remove(cell_gama)
                        gama_cell_temp_index_of_remove.append(gama_index_temp)
                    gama_index_temp += 1
                    
                # remove cell_beta from LS_filtra and LS_ele
                for i in reversed(gama_cell_temp_index_of_remove):
                    del LS_filtra[i]
                    del LS_ele[i]
                        
                PQone = sorted(PQone)
                PQone = deque(PQone) # after sorting, PQone one will be a list instead of a queue, we need to reconstruct this queue
                
    return crit_cell, Forman_vec_pair_VE, Forman_vec_pair_ET

# StructType for Crit_cell        
add_Crit_cell_schema = StructType([
    StructField('Crit_cell',ArrayType(IntegerType()),True)
])

# StructType for VE pairs        
add_VE_schema = StructType([
    StructField("VE_pair_V", IntegerType(),True), 
    StructField('VE_pair_E',ArrayType(IntegerType()),True)
])

# StructType for ET pairs        
add_multi_ET_schema = StructType([
    StructField("ET_pair_E", ArrayType(IntegerType()),True), 
    StructField('ET_pair_T',ArrayType(IntegerType()),True)
])
 
# the whole StructType
get_Forman_schema = StructType([
    StructField("Crit_cell", ArrayType(ArrayType(IntegerType())), True),
    StructField("VE_pair", ArrayType(add_VE_schema), True),
    StructField("ET_pair", ArrayType(add_multi_ET_schema), True)
])

# convert a function to an udf and determine the return type
# https://spark.apache.org/docs/3.1.3/api/python/reference/api/pyspark.sql.functions.udf.html
get_Forman_udf = udf(get_Forman, get_Forman_schema)

In [19]:
df_Forman = df_LS.withColumn("LS_Forman", get_Forman_udf(df_LS.Ver, df_LS.LS_edge, df_LS.LS_tri_filtra, df_LS.LS_tri_ele))
df_Forman = df_Forman.select(col("Ver"), col("VT_filtra"), col("LS_Forman").alias("Forman"))
df_Forman.printSchema()
print("The number of partitions for df_Forman:", df_Forman.rdd.getNumPartitions())
# df_Forman.show()

root
 |-- Ver: integer (nullable = true)
 |-- VT_filtra: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- Forman: struct (nullable = true)
 |    |-- Crit_cell: array (nullable = true)
 |    |    |-- element: array (containsNull = true)
 |    |    |    |-- element: integer (containsNull = true)
 |    |-- VE_pair: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- VE_pair_V: integer (nullable = true)
 |    |    |    |-- VE_pair_E: array (nullable = true)
 |    |    |    |    |-- element: integer (containsNull = true)
 |    |-- ET_pair: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- ET_pair_E: array (nullable = true)
 |    |    |    |    |-- element: integer (containsNull = true)
 |    |    |    |-- ET_pair_T: array (nullable = true)
 |    |    |    |    |-- element: integer (containsNull = true)

The number of par

In [20]:
# # for validation purpose
# df_LS_col = df_LS.collect()

# t0 = time.time()
# for i in range(len(df_LS_col)):
#     if i % 100000 == 0:
#         t1 = time.time()
        
#         print("i = ", i)
#         print("time cost:", t1 - t0)
#     Ver, LS_edge, LS_filtra, LS_ele = df_LS_col[i]['Ver'], df_LS_col[i]['LS_edge'], df_LS_col[i]['LS_tri_filtra'], df_LS_col[i]['LS_tri_ele']
#     try:
#         return_1, return_2, return_3 = get_Forman(Ver, LS_edge, LS_filtra, LS_ele)
#     except Exception as e:
#         print(f"❌ Error at i = {i}: {e}")
#         # Optionally, skip to the next iteration
#         break

### save df_Forman

In [21]:
t0 = time.time()

file_df_Forman = directory + '/' + tin_filename + '_sortForman' + '.parquet'
df_Forman.write.parquet(file_df_Forman) 

t1 = time.time()
print("time cost:", t1-t0)

time cost: 841.9455440044403


### get Forman gradient

In [22]:
file_df_Forman = directory + '/' + tin_filename + '_sortForman' + '.parquet'

df_Forman = spark.read.parquet(file_df_Forman)
df_Forman.printSchema()

root
 |-- Ver: integer (nullable = true)
 |-- VT_filtra: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- Forman: struct (nullable = true)
 |    |-- Crit_cell: array (nullable = true)
 |    |    |-- element: array (containsNull = true)
 |    |    |    |-- element: integer (containsNull = true)
 |    |-- VE_pair: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- VE_pair_V: integer (nullable = true)
 |    |    |    |-- VE_pair_E: array (nullable = true)
 |    |    |    |    |-- element: integer (containsNull = true)
 |    |-- ET_pair: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- ET_pair_E: array (nullable = true)
 |    |    |    |    |-- element: integer (containsNull = true)
 |    |    |    |-- ET_pair_T: array (nullable = true)
 |    |    |    |    |-- element: integer (containsNull = true)



In [23]:
df_Forman.show()

+----+--------------------+--------------------+
| Ver|           VT_filtra|              Forman|
+----+--------------------+--------------------+
| 192|[[2604, 1096, 192...|   [[[192]], [], []]|
| 327|[[327, 174, 93], ...|[[], [[327, [327,...|
| 390|[[4294, 2975, 390...|   [[[390]], [], []]|
| 420|[[23983, 6102, 42...|   [[[420]], [], []]|
| 682|[[1846, 1310, 682...|   [[[682]], [], []]|
|1098|[[5587, 1141, 109...|  [[[1098]], [], []]|
|1703|[[16947, 3038, 17...|  [[[1703]], [], []]|
|1766|[[2567, 2034, 176...|  [[[1766]], [], []]|
|1910|[[192052, 8573, 1...|  [[[1910]], [], []]|
|1992|[[12561, 1992, 22...|[[[1992, 1136]], ...|
|2026|[[2319, 2026, 128...|[[[2026, 1480]], ...|
|2195|[[7424, 2447, 219...|  [[[2195]], [], []]|
|2379|[[7422, 2379, 171...|[[], [[2379, [237...|
|2458|[[1242512, 43175,...|  [[[2458]], [], []]|
|2805|[[2805, 2404, 591...|[[], [[2805, [280...|
|2854|[[2854, 1817, 156...|[[], [[2854, [285...|
|2943|[[8027, 2943, 158...|[[], [[2943, [294...|
|3161|[[37413567, 21